In [15]:
%load_ext autoreload
%autoreload 2

import re
import numpy as np
from models.odor_strength_module import OdorStrengthModuleHyperparameterOptimizationWrapper
from data.molecules.smiles_converter import SmilesCanonicalizer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
class Ensemble:
    def __init__(self, models: list):
        self.models = models
        self.smiles_canonicalizer = SmilesCanonicalizer()

    def fit(self, X, y):
        X = self.smiles_canonicalizer.canonicalize_smiles(X)
        for model in self.models:
            model.fit(X, y)

    def predict(self, X):
        X = self.smiles_canonicalizer.canonicalize_smiles(X)
        preds = np.array([model.predict(X) for model in self.models])
        return np.mean(preds, axis=0)
    
    def save(self, encoder_paths: list[str], predictor_paths: list[str], predictor_hyperparameters_paths: list[str]):
        for model, encoder_path, predictor_path, hyperparameters_path in zip(self.models, encoder_paths, predictor_paths, predictor_hyperparameters_paths):
            model.save(encoder_path, predictor_path, hyperparameters_path)

    def load(self, encoder_paths: list[str], predictor_paths: list[str], predictor_hyperparameters_paths: list[str]):
        for i, (encoder_path, predictor_path, hyperparameters_path) in enumerate(zip(encoder_paths, predictor_paths, predictor_hyperparameters_paths)):
            self.models[i].load(encoder_path, predictor_path, hyperparameters_path)

DIRECTORY = 'models/application_ensemble_model/'
encoder_paths = [
    DIRECTORY + 'rdkit_descriptors_encoder.gz',
]*3
predictor_paths = [
    DIRECTORY + 'random_forest_predictor.pt',
    DIRECTORY + 'xbgoost_predictor.pt',
    DIRECTORY + 'mlp_predictor.pt',
]

predictor_hyperparameter_paths = [
    DIRECTORY + 'random_forest_predictor_hyperparameters.json',
    DIRECTORY + 'xbgoost_predictor_hyperparameters.json',
    DIRECTORY + 'mlp_predictor_hyperparameters.json',
]

with open(DIRECTORY + 'odor_strength_model_config.txt', 'r') as f:
    model_config = f.read()
    encoder_name_1 = re.search(r'Encoder_1: (.+)', model_config).group(1)
    predictor_name_1 = re.search(r'Predictor_1: (.+)', model_config).group(1)
    encoder_name_2 = re.search(r'Encoder_2: (.+)', model_config).group(1)
    predictor_name_2 = re.search(r'Predictor_2: (.+)', model_config).group(1)
    encoder_name_3 = re.search(r'Encoder_3: (.+)', model_config).group(1)
    predictor_name_3 = re.search(r'Predictor_3: (.+)', model_config).group(1)
    encoder_names = [encoder_name_1, encoder_name_2, encoder_name_3]
    predictor_names = [predictor_name_1, predictor_name_2, predictor_name_3]

models = []
for encoder_name, predictor_name, encoder_path, predictor_path, predictor_hyperparameter_path in zip(encoder_names, predictor_names, encoder_paths, predictor_paths, predictor_hyperparameter_paths):
    model = OdorStrengthModuleHyperparameterOptimizationWrapper(
        encoder_name=encoder_name,
        predictor_name=predictor_name,
    )
    model.load(
        encoder_path=encoder_path,
        predictor_path=predictor_path,
        predictor_hyperparameter_path=predictor_hyperparameter_path
    )
    models.append(model)

ensemble_model = Ensemble(models=models)
# model = OdorStrengthModuleHyperparameterOptimizationWrapper(
#     encoder_name=encoder_name_1,
#     predictor_name=predictor_name_1,
#     )
# model.load(
#     encoder_path='models/application_ensemble_model/rdkit_descriptors_encoder.gz',
#     predictor_path='models/application_ensemble_model/state_dict.pt',
#     predictor_hyperparameter_path='models/application_ensemble_model/odor_strength_predictor_hyperparameters.json'
# )

Using device: cuda
Using device: cuda


In [19]:
# 0 odorless
# 1 low odor strength
# 2 medium odor strength
# 3 high odor strength
ensemble_model.predict(['CC', 'CCCCCO', 'CCCCCCCC=O', 'CC(=CCCC(C)(C=C)O)C'])

array([0.33333333, 2.66666667, 3.        , 2.        ])